# 🛣️ Smart City Pothole & Road Defect Detection — YOLOv8 Training Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lokitheeditor697-create/Pathole/blob/main/train_pothole_yolov8.ipynb)

This pipeline trains a custom **YOLOv8** model on verified road pothole datasets for real-time edge processing.

### Step 1: Install Ultralytics & Check GPU

In [ ]:
!nvidia-smi
!pip install -q ultralytics matplotlib opencv-python

### Step 2: Download Verified Pothole Dataset

In [ ]:
import os

# Clean previous runs
!rm -rf /content/road_defect_data

# Clone verified public Pothole dataset from Hugging Face (100% active, free, no API key needed)
!git clone https://huggingface.co/datasets/Ryukijano/Pothole-detection-Yolov8 /content/road_defect_data

# Set clean data.yaml with absolute Colab paths
yaml_path = "/content/road_defect_data/data.yaml"
with open(yaml_path, "w") as f:
    f.write("""
path: /content/road_defect_data
train: train/images
val: valid/images
test: test/images

names:
  0: pothole
""")

print("✅ Verified Pothole Dataset Ready!")
print("Train images found:", len(os.listdir('/content/road_defect_data/train/images')))
print("Val images found:", len(os.listdir('/content/road_defect_data/valid/images')))

### Step 3: Train YOLOv8 Model on Tesla T4 GPU

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 Nano model (transfer learning)
model = YOLO('yolov8n.pt')

# Train for 50 epochs
results = model.train(
    data='/content/road_defect_data/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=10,
    name='pothole_yolov8_model'
)

### Step 4: Evaluate Metrics (mAP50, Loss & Confusion Matrix)

In [ ]:
import matplotlib.pyplot as plt
import cv2

# Run validation
metrics = model.val()
print(f"Overall mAP50: {metrics.box.map50:.4f}")
print(f"Overall mAP50-95: {metrics.box.map:.4f}")

# Display training loss curves & metrics
results_img = cv2.imread('runs/detect/pothole_yolov8_model/results.png')
if results_img is not None:
    plt.figure(figsize=(16, 10))
    plt.imshow(cv2.cvtColor(results_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Training Loss & Metrics')
    plt.show()

### Step 5: Test Model on Test Images

In [ ]:
# Run prediction on test set
preds = model.predict(source='/content/road_defect_data/test/images', conf=0.5, save=True)
print("Predictions saved to runs/detect/predict/")

### Step 6: Download Trained Model (`best.pt`)

In [ ]:
from google.colab import files
files.download('runs/detect/pothole_yolov8_model/weights/best.pt')